In [1]:
import numpy as np
import cv2 as cv
from ultralytics import YOLO
import time
from pathlib import Path

In [2]:
pr_dir = Path.cwd().parents[0]
path_model = pr_dir / 'models' / 'yolo11n-pose.pt'
model = YOLO(path_model)

In [17]:
cap = cv.VideoCapture(0)

if not cap.isOpened():
    print('Камера недоступна')

time_t = time.time()
_ , frame2 = cap.read()

while cap.isOpened():
    ret, frame = cap.read()

    frame = cv.resize(frame, (448, 448))
    
    #frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

    if not ret:
        break
      


    if cv.waitKey(1) & 0xFF == ord('p'):
        cv.imwrite('hand.png', frame)
    
    if time.time() - time_t >= 1:
        time_t = time.time()
        frame2 = frame.copy()
        results = model(
                    frame, 
                    stream = False, 
                    #vid_stride = 25,
                    verbose = False)
        for result in results:
            xy = result.keypoints.xy
            for x,y in xy.reshape(-1,2):
                
                cv.circle(frame2, (int(x),int(y)),3,(255,0,0),-1) #rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
       
    cv.imshow('frame2', frame2)


    #cv.imshow('frame', frame)

cap.release()
            
cv.destroyAllWindows()

In [18]:
for i, result in enumerate(results):
    xy = result.keypoints.xy  # x and y coordinates
    xyn = result.keypoints.xyn  # normalized
    kpts = result.keypoints.data  # x, y, visibility (if available)
    print(f'Результат {i}: {kpts}')

Результат 0: tensor([[[2.9766e+02, 1.3925e+02, 9.9881e-01],
         [3.0036e+02, 1.2080e+02, 8.5829e-01],
         [2.7814e+02, 1.1338e+02, 9.9931e-01],
         [2.7331e+02, 1.2724e+02, 1.3552e-03],
         [2.0999e+02, 1.1782e+02, 9.9790e-01],
         [2.4787e+02, 2.5929e+02, 9.9422e-01],
         [1.3441e+02, 2.7221e+02, 8.9705e-01],
         [2.9932e+02, 3.5367e+02, 8.9144e-01],
         [1.1516e+02, 4.2273e+02, 1.1027e-01],
         [3.9153e+02, 3.4435e+02, 9.3470e-01],
         [2.6165e+02, 4.1838e+02, 4.7737e-01],
         [2.2865e+02, 4.4800e+02, 4.0962e-02],
         [1.5562e+02, 4.4800e+02, 4.8295e-03],
         [3.2444e+02, 4.0562e+02, 1.7411e-03],
         [2.5793e+02, 4.3285e+02, 2.1897e-04],
         [3.3856e+02, 3.8540e+02, 1.6771e-04],
         [2.9791e+02, 4.0434e+02, 2.4435e-05]]])


In [22]:
from PIL import Image
for i, r in enumerate(results):
    # Plot results image
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    r.show()

In [23]:
from ultralytics import YOLO

# Open the video file
video_path = 0
cap = cv.VideoCapture(video_path)

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the frame
        results = model(frame)

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        # Display the annotated frame
        cv.imshow("YOLO Inference", annotated_frame)

        # Break the loop if 'q' is pressed
        if cv.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv.destroyAllWindows()